# Segmentación Estratégica de Barrios (Clustering)
# Barcelona Housing Market Analysis

**Objetivo:** Agrupar los barrios de Barcelona según sus características inmobiliarias y socioeconómicas para identificar perfiles de inversión y zonas con comportamientos similares.

---

## Contenido
1. [Configuración y Carga de Datos](#1-configuracion)
2. [Ingeniería de Características para Clustering](#2-features)
3. [Preprocesamiento y Escalado](#3-prepro)
4. [Determinación del Número de Clústeres (K)](#4-elbow)
5. [Aplicación de K-Means](#5-kmeans)
6. [Interpretación y Perfiles de Clústeres](#6-interpretacion)
7. [Visualización Geo-Estratégica](#7-viz)
8. [Conclusiones](#8-conclusiones)

## 1. Configuración y Carga de Datos <a name="1-configuracion"></a>

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

# Conexión a la base de datos
db_path = Path("../data/master.db")
conn = sqlite3.connect(db_path)

print(f"✅ Conectado a {db_path}")

## 2. Ingeniería de Características para Clustering <a name="2-features"></a>
Para segmentar los barrios, utilizaremos los datos más recientes disponibles (2023) integrando múltiples dimensiones.

In [ ]:
# Cargamos datos de 2023 integrados
query = """
SELECT 
    b.barrio_id,
    b.barrio_nombre,
    b.distrito_nombre,
    p.precio_m2_venta,
    p.precio_mes_alquiler,
    r.renta_bruta_llar,
    r.indice_gini,
    h.promedio_personas_por_hogar,
    c.superficie_media_m2,
    c.antiguedad_media_bloque
FROM dim_barrios b
LEFT JOIN fact_precios p ON b.barrio_id = p.barrio_id AND p.anio = 2023
LEFT JOIN fact_renta_avanzada r ON b.barrio_id = r.barrio_id AND r.anio = 2023
LEFT JOIN fact_hogares_avanzado h ON b.barrio_id = h.barrio_id AND h.anio = 2023
LEFT JOIN fact_catastro_avanzado c ON b.barrio_id = c.barrio_id AND c.anio = 2023
"""

df_raw = pd.read_sql(query, conn)

# Cálculo del Yield Bruto (Rentabilidad)
# Suponiendo piso de 80m2 para la relación alquiler/venta
df_raw['yield_bruto'] = ((df_raw['precio_mes_alquiler'] * 12) / (df_raw['precio_m2_venta'] * 80)) * 100

# Limpieza: barrios con datos suficientes
df_cluster = df_raw.dropna(subset=['precio_m2_venta', 'renta_bruta_llar', 'yield_bruto'])

print(f"📊 Barrios para clustering: {len(df_cluster)}")
df_cluster.head()

## 3. Preprocesamiento y Escalado <a name="3-prepro"></a>

In [ ]:
# Seleccionamos variables numéricas para el modelo
features = [
    'precio_m2_venta', 
    'yield_bruto', 
    'renta_bruta_llar', 
    'indice_gini', 
    'superficie_media_m2', 
    'antiguedad_media_bloque'
]

X = df_cluster[features]

# Escalado estándar (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Datos escalados y listos")

## 4. Determinación del Número de Clústeres (K) <a name="4-elbow"></a>

In [ ]:
# Método del Codo (Elbow Method)
wcss = []
silhouette_avg = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_avg.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualizar resultados
fig, ax1 = plt.subplots()

ax1.plot(k_range, wcss, 'bx-')
ax1.set_xlabel('Número de Clústeres (k)')
ax1.set_ylabel('Inercia (WCSS)', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_avg, 'ro-')
ax2.set_ylabel('Coeficiente de Silhouette', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Selección de K óptimo: Codo y Silhouette')
plt.show()

## 5. Aplicación de K-Means <a name="5-kmeans"></a>
Basado en los gráficos anteriores sugerimos un **K=4** para capturar la diversidad sin sobre-segmentar.

In [ ]:
k_optimo = 4
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

print(f"✅ Clústeres asignados: {df_cluster['cluster'].value_counts().to_dict()}")

## 6. Interpretación y Perfiles de Clústeres <a name="6-interpretacion"></a>

In [ ]:
# Analizamos promedios por clúster
cluster_profile = df_cluster.groupby('cluster')[features].mean()
cluster_profile['count'] = df_cluster['cluster'].value_counts()

print("📊 Perfil promedio de cada clúster:")
display(cluster_profile.sort_values('precio_m2_venta', ascending=False))

In [ ]:
# Visualización de perfiles con boxplots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.boxplot(x='cluster', y=feat, data=df_cluster, ax=axes[i])
    axes[i].set_title(f'Distribución de {feat}')

plt.tight_layout()
plt.show()

## 7. Visualización Geo-Estratégica <a name="7-viz"></a>

In [ ]:
# Scatter plot: Precio vs Yield coloreado por clúster
plt.figure(figsize=(12, 8))
scatter = sns.scatterplot(
    data=df_cluster, 
    x='precio_m2_venta', 
    y='yield_bruto', 
    hue='cluster', 
    size='renta_bruta_llar',
    sizes=(20, 400), 
    palette='viridis', 
    alpha=0.7
)

plt.title('Matriz Estratégica: Precio vs Rentabilidad por Clúster')
plt.xlabel('Precio Venta (€/m²)')
plt.ylabel('Yield Bruto (%)')
plt.legend(title='Clúster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

## 8. Conclusiones <a name="8-conclusiones"></a>

### Definición de Segmentos:
- **Clúster 0 (Premium):** Barrios con mayor renta y precios más altos. Menor rentabilidad por alquiler pero mayor potencial de apreciación de capital.
- **Clúster 1 (Inversión/Oportunidad):** Precios bajos y rentabilidad (Yield) superior al promedio. Zonas periféricas con buen retorno inmediato.
- **Clúster 2 (Estable/Clase Media):** Barrios con métricas balanceadas.
- **Clúster 3 (Histórico/Antiguo):** Barrios con alta antigüedad de construcción y superficies medias menores.

**Próximo Paso:** Exportar etiquetas de clústeres para integrarlas en el modelo predictivo de precios (Fase 4).

In [ ]:
# Guardar resultados
output_path = "../data/neighborhood_clusters.csv"
df_cluster[['barrio_id', 'barrio_nombre', 'cluster']].to_csv(output_path, index=False)
print(f"✅ Resultados exportados a {output_path}")
conn.close()